# Phase 3 (Stage 2) -- GNN + XGBoost ensemble

**Inputs:**
* `data/graphs/hetero.pt`            -- Phase 2 heterograph
* `data/graphs/features.parquet`     -- 119 engineered features (Phase 2)
* `data/models/gnn/state_dict.pt`    -- trained GNN from notebook 03

**What it does:**
1. Forward the GNN to extract a 64-d embedding per transaction.
2. Concat with the 119 tabular features.
3. Fit XGBoost (n_est=500, depth=8, lr=0.05, scale_pos_weight=27.6, aucpr eval).
4. Score val + test, render PR / ROC / calibration plots.

**Plan target:** ensemble val AUPRC > 0.70.

In [ ]:
import sys
from pathlib import Path

# Resolve project root robustly: works whether the notebook is launched from the
# repo root (cwd has src/) or from notebooks/ (cwd's parent has src/).
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch_geometric.transforms import ToUndirected

from fraud_detection.utils.config import load_config
from fraud_detection.models import FraudHeteroGNN, FraudEnsemble, XGBoostFraudModel, XGBoostConfig
from fraud_detection.training import (
    ensure_temporal_masks, evaluate_predictions, write_evaluation_report,
)

sns.set_theme(style='whitegrid', context='talk')
cfg = load_config()
print(f'project root: {ROOT}')

## 1. Load Phase 2 + stage-1 artefacts

In [ ]:
data = torch.load(cfg.paths.data_graphs / 'hetero.pt', weights_only=False)
for nt in data.node_types:
    data[nt].x = data[nt].x.float()
data = ToUndirected()(data)
data = ensure_temporal_masks(data, target_node_type='transaction')
features = pd.read_parquet(cfg.paths.data_graphs / 'features.parquet')

node_feature_dims = {nt: data[nt].num_node_features for nt in data.node_types}
gnn = FraudHeteroGNN(node_feature_dims=node_feature_dims, edge_types=data.edge_types)
gnn.load_state_dict(torch.load(ROOT / 'data/models/gnn/state_dict.pt', weights_only=True))
gnn.eval()
print(f'features: {features.shape}, GNN params: {gnn.n_parameters():,}')

## 2. Slice splits + fit ensemble

In [ ]:
tx = data['transaction']
train_idx = tx.train_mask.nonzero(as_tuple=True)[0]
val_idx = tx.val_mask.nonzero(as_tuple=True)[0]
test_idx = tx.test_mask.nonzero(as_tuple=True)[0]

def _tab(mask_idx):
    df = features.iloc[mask_idx.numpy()].reset_index(drop=True)
    drop = [c for c in ('TransactionID', 'isFraud', 'event_timestamp') if c in df.columns]
    return df.drop(columns=drop)

tab_train, tab_val, tab_test = _tab(train_idx), _tab(val_idx), _tab(test_idx)
y_train = tx.y[train_idx].numpy().astype(np.int8)
y_val = tx.y[val_idx].numpy().astype(np.int8)
y_test = tx.y[test_idx].numpy().astype(np.int8)

ensemble = FraudEnsemble(gnn=gnn, xgboost_model=XGBoostFraudModel(XGBoostConfig()))
ensemble.fit_xgboost(
    train_data=data, train_indices=train_idx,
    train_tabular=tab_train.to_numpy(np.float32), train_y=y_train,
    tabular_columns=list(tab_train.columns),
    val_data=data, val_indices=val_idx,
    val_tabular=tab_val.to_numpy(np.float32), val_y=y_val,
)
print(f'ensemble feature space: {len(ensemble.feature_columns)} cols ' 
      f'(GNN_emb={len(ensemble.gnn_embedding_columns)} + tabular={len(tab_train.columns)})')

## 3. Score and compare with GNN-only

In [ ]:
# Ensemble scores
ens_val = ensemble.predict_proba(data, tab_val.to_numpy(np.float32), target_indices=val_idx)
ens_test = ensemble.predict_proba(data, tab_test.to_numpy(np.float32), target_indices=test_idx)

# GNN-only scores for comparison
with torch.no_grad():
    gnn_logits_all = gnn(data)
gnn_probs = torch.sigmoid(gnn_logits_all).numpy()
gnn_val = gnn_probs[val_idx.numpy()]
gnn_test = gnn_probs[test_idx.numpy()]

rows = []
for name, y, scores in [
    ('GNN only / val', y_val, gnn_val),
    ('Ensemble / val', y_val, ens_val),
    ('GNN only / test', y_test, gnn_test),
    ('Ensemble / test', y_test, ens_test),
]:
    res = evaluate_predictions(y, scores)
    rows.append({'model': name, 'AUPRC': res.auprc, 'AUROC': res.auroc, 'best_F1': res.best_f1})
pd.DataFrame(rows).set_index('model').round(4)

## 4. Feature importance (top 30)

In [ ]:
imp = ensemble.xgb.feature_importance(kind='gain')
imp_series = pd.Series(imp).sort_values(ascending=False).head(30)

fig, ax = plt.subplots(figsize=(8, 9))
imp_series[::-1].plot.barh(ax=ax, color='#c44e52', edgecolor='white')
ax.set_xlabel('XGBoost gain')
ax.set_title('Top-30 features by XGBoost gain')
plt.tight_layout()
n_emb = sum(1 for c in imp_series.index if c.startswith('gnn_emb_'))
print(f'top-30 includes {n_emb} GNN embedding dims and {len(imp_series) - n_emb} tabular features')

## 5. PR + calibration overlay (GNN vs ensemble)

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for label, scores, color in [('GNN', gnn_val, '#4c72b0'), ('Ensemble', ens_val, '#c44e52')]:
    p, r, _ = precision_recall_curve(y_val, scores)
    ap = average_precision_score(y_val, scores)
    axes[0].plot(r, p, lw=2, label=f'{label} (AUPRC={ap:.3f})', color=color)
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision'); axes[0].set_title('PR curve (val)'); axes[0].legend(); axes[0].grid(alpha=0.3)

for label, scores, color in [('GNN', gnn_val, '#4c72b0'), ('Ensemble', ens_val, '#c44e52')]:
    frac_pos, mean_pred = calibration_curve(y_val, scores, n_bins=10, strategy='quantile')
    axes[1].plot(mean_pred, frac_pos, 'o-', lw=2, label=label, color=color)
axes[1].plot([0, 1], [0, 1], '--', color='grey', alpha=0.5, label='perfect')
axes[1].set_xlabel('Mean predicted prob'); axes[1].set_ylabel('Fraction positive'); axes[1].set_title('Calibration (val)'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()